# Scenario-grouped evaluation and controlled head comparison

Extension of the canonical Meeting 2 notebook, credited to the late
[ACM AI – Poker](https://colab.research.google.com/drive/1GHX6o-6mzOQirpwDJlrc1yZ5bD1CBCCC)
and [day2_colab](https://colab.research.google.com/drive/1M-aPyxDIUkv9k3zTd2PkASfhA2cSn8pG)
submissions. Our model stays a **joint all-options encoder**. The source MiniLM
results used context/candidate pairs; they are not results of this implementation.

Use a local CUDA GPU or Colab GPU for training. **RUN_CV is off initially:** first inspect and freeze
the protocol. Set it to True to execute. Both heads use identical folds, backbone,
epochs, batch size and learning rates. Each fold fits a fresh menu prior, linear
baseline, encoder, head and optimizer. Held-out folds never select checkpoints.

The pool has already been inspected. This is development CV, not a pristine final
benchmark. Later choices informed by these scores require inner folds or a new test.
No current notebook demonstrates unexploitable poker play. No calibration is fitted here.

For the complete four-encoder/two-head GPU matrix outside Jupyter, see
`docs/week2-training.md`. The runner preserves the grouped protocol and requires
training-only tiny-set learning checks before full fits.


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None
)
IN_COLAB = (
    importlib.util.find_spec("google.colab") is not None
    if importlib.util.find_spec("google")
    else False
)
if ROOT is None and IN_COLAB:
    ROOT = Path("/content/Poker-Decision-Model")
    if not ROOT.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/nmokey/Poker-Decision-Model.git", str(ROOT)],
            check=True,
        )
    os.chdir(ROOT)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the project checkout.")
if not (ROOT / "src/poker_decision/training.py").is_file():
    raise RuntimeError("Use a checkout containing the Meeting 2 changes, then rerun.")
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[analysis,ml]"], check=True
    )
print("Project:", ROOT)

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

from poker_decision.cross_validation import CVConfig, run_cross_validation, sweep_configs
from poker_decision.diagnostics import menu_information
from poker_decision.pipeline import prepare
from poker_decision.scenarios import digest, make_folds, scenario_overlap
from poker_decision.training import load_prepared

RUN_CV = False  # Inspect the protocol, then enable on a CUDA GPU.
# Single-protocol knobs: use any of bert_mini, minilm, distilbert, modernbert.
CONFIG = CVConfig(
    backbone="bert_mini",
    heads=("linear", "mlp"),  # Each head is a separate fresh model fit.
    head_width=64,  # MLP only
    n_splits=5,
    seed=42,
    epochs=1,  # Fixed in advance; held-out folds never choose the checkpoint.
    batch_size=16,
    max_length=512,
    encoder_lr=2e-5,
    head_lr=1e-3,
    weight_decay=0.01,
)
# Optional: point OUTPUT at your already-mounted Drive folder for runtime survival.
OUTPUT = ROOT / "artifacts/scenario_evaluation"
OUTPUT.mkdir(parents=True, exist_ok=True)
if not (ROOT / "data/processed/pokerbench_meeting1/manifest.json").exists():
    prepare(ROOT)
train, validation, manifest = load_prepared(ROOT)
print("Historical holdout overlap:", scenario_overlap(train, validation))
pool, folds, protocol = make_folds(train + validation, n_splits=CONFIG.n_splits, seed=CONFIG.seed)
(OUTPUT / "preview_protocol.json").write_text(json.dumps(protocol, indent=2) + "\n")
print("Pool fingerprint:", protocol["pool_sha256"])
print("Protocol preview fingerprint:", digest(protocol))
display(
    pd.DataFrame(
        [
            {
                k: f[k]
                for k in ["fold", "train_n", "test_n", "scenario_overlap", "exact_context_overlap"]
            }
            for f in protocol["folds"]
        ]
    )
)

## What is held out?

Preflop groups share the full previous betting history (including amounts and seats).
Postflop groups share the sorted ranks of the first three board cards, including suit
variants and all later streets. Grouping is answer-free. Approximate stratification
uses street and label type only to allocate folds; neither becomes a model feature.
A group may be large; inspect per-street counts. This is a particular stress test of
scenario transfer, not proof that every kind of related hand or session is isolated.

The historical split and CV pool have different denominators: 2,000 validation rows
versus 10,000 out-of-fold rows. Don't interpret changed scores as a controlled model
improvement. The two heads below ARE compared on identical rows/settings.

In [ ]:
# Optional diagnostic only on training rows of the first frozen fold.
first_fit = [pool[i] for i in folds[0][0]]
menu_report = menu_information(first_fit, permutations=199, seed=CONFIG.seed)
print(menu_report)
(OUTPUT / "training_only_menu_information.json").write_text(
    json.dumps(menu_report, indent=2) + "\n"
)

## Execute the fixed protocol on a CUDA GPU

The runner saves the complete protocol before fitting. Only completed folds with
matching protocol, prediction and checkpoint hashes may be reused. Resume the same
folder/configuration after reconnecting; incomplete folds retrain. Changing data,
code, package versions, device or settings creates a distinct protocol folder.

Do not select epochs or a winning configuration and quote the same outer-fold scores
as an unbiased final estimate. Fold loss SD is descriptive, not a confidence interval.
Paired bootstrap intervals are conditional summaries of saved fold predictions.

In [ ]:
if RUN_CV:
    if not torch.cuda.is_available():
        raise RuntimeError("Pretrained training requires a CUDA GPU runtime.")
    result = run_cross_validation(pool, OUTPUT, config=CONFIG, device="cuda")
    display(pd.DataFrame(result["models"]).T)
    print(result["interpretation"])
    print("Saved evidence:", result["output_directory"])
else:
    print("Protocol checked; no pretrained model loaded or trained. Enable RUN_CV on a CUDA GPU.")

## Inspect the evidence

Each model gets exactly one held-out prediction per source row. The summary contains
row-weighted pooled metrics, mean/SD of fold loss, per-street metrics, reliability
bins/ECE, confusion counts, sizing errors, paired comparisons and five mistakes.
Pooled loss is independently checked against the row-weighted fold losses. Metrics
are recomputed before accepting resumed reports. Download the complete output folder
before Colab ends unless you used persistent storage. The official test remains outside
this workflow; other team notebooks may already have inspected it.

In [ ]:
if RUN_CV:
    import matplotlib.pyplot as plt

    for name, diagnostic in result["diagnostics"].items():
        bins = pd.DataFrame(diagnostic["reliability"])
        plt.plot(bins["mean_confidence"], bins["accuracy"], marker="o", label=name)
    plt.plot([0, 1], [0, 1], "--", color="gray")
    plt.xlabel("Mean maximum reference-label confidence")
    plt.ylabel("Exact action accuracy")
    plt.title("Out-of-fold reliability; bin counts are in summary.json")
    plt.legend()
    plt.show()
    display(pd.DataFrame(result["paired_vs_linear"]).T)
    for name, errors in result["five_mistakes"].items():
        print(name)
        display(pd.DataFrame(errors))

## Optional development sweeps — preview first

Edit `GRID`, inspect the table and fit count, then enable `RUN_SWEEP` on a CUDA GPU.
Keep `RUN_CV=False` when running a sweep to avoid a separate single-protocol run.
Every grid axis is a list; the Cartesian product creates trials. For head choices use
`"heads": [("linear",), ("mlp",)]`; `head_width` affects only MLPs. You can also vary
`epochs`, `batch_size`, `max_length`, `head_lr`, and `weight_decay`. Start with one axis.
The explicit trial budget prevents accidental huge grids. Runs are sequential,
freshly initialized, and resume from separate hashed protocol folders.

**Interpretation:** these are exploratory comparisons on the already inspected
development pool. Do not tune settings or epochs using a held-out fold during its
fit. Selecting a winner after seeing these results makes its reported score a tuning
score. For a final estimate, freeze the choice and use an untouched test set, or
implement nested grouped CV with selection confined to each outer training fold.
This sweep does not perform nested selection or produce a final winner estimate.
Seed and fold count stay fixed across trials; each model uses its own tokenizer.
Same epochs/batches mean the same training schedule, not the same compute cost.
If you hit GPU memory limits, lower `batch_size` in the recorded grid and rerun.


In [ ]:
from dataclasses import asdict

RUN_SWEEP = False
MAX_TRIALS = 8
GRID = {
    "backbone": ["bert_mini", "minilm", "distilbert", "modernbert"],
    # Add one axis at a time, e.g. "encoder_lr": [1e-5, 2e-5].
}
TRIALS = sweep_configs(CONFIG, GRID, max_trials=MAX_TRIALS)
display(pd.DataFrame([asdict(config) for config in TRIALS]))
print("Trials:", len(TRIALS))
print("Transformer fits:", sum(c.n_splits * len(c.heads) for c in TRIALS))
print("Fixed epochs per fit:", [c.epochs for c in TRIALS])
print("Development comparison only; no automatic winner selection.")

In [ ]:
if RUN_SWEEP:
    if RUN_CV:
        raise ValueError("Set RUN_CV=False to run only the sweep.")
    if not torch.cuda.is_available():
        raise RuntimeError("Pretrained training requires a CUDA GPU runtime.")
    sweep_rows = []
    for trial_index, config in enumerate(TRIALS):
        print(f"Trial {trial_index + 1}/{len(TRIALS)}:", config)
        trial_result = run_cross_validation(pool, OUTPUT, config=config, device="cuda")
        for model_name, metrics in trial_result["models"].items():
            sweep_rows.append(
                {
                    "trial": trial_index + 1,
                    "backbone": config.backbone,
                    "model": model_name,
                    **metrics,
                    "output_directory": trial_result["output_directory"],
                }
            )
    sweep_table = pd.DataFrame(sweep_rows)
    display(sweep_table)
    sweep_table.to_csv(
        OUTPUT / f"development_sweep-{digest([asdict(c) for c in TRIALS])[:12]}.csv", index=False
    )
else:
    print("Sweep preview only; no pretrained model loaded or trained.")

## Completed PokerBench experiment — November 2, 2026

All four pinned pretrained encoders and both decision heads completed five fixed
scenario-grouped folds with three epochs per fit: 40 full fits and 10,000 held-out
predictions per model. See [all accuracy and log-loss scores](../docs/week2-results.md),
[CSV](../docs/results/week2-development.csv), and
[recorded settings/provenance](../docs/results/week2-development.json).

| Transformer | Head | Accuracy | Log loss (lower is better) |
| --- | --- | ---: | ---: |
| BERT Mini | Linear | 68.96% | 0.6722 |
| BERT Mini | MLP | 69.35% | 0.6656 |
| MiniLM | Linear | 69.57% | 0.6849 |
| MiniLM | MLP | 68.83% | 0.6764 |
| DistilBERT | Linear | 71.78% | 0.6164 |
| DistilBERT | MLP | 71.60% | 0.6213 |
| ModernBERT | Linear | 71.92% | 0.6078 |
| ModernBERT | MLP | 71.38% | 0.6158 |
| Linear candidate baseline | — | 69.69% | 1.0025 |

These are one-seed development comparisons on an inspected pool. The small
ModernBERT/DistilBERT gap does not establish a reliable advantage; the scores do
not show profitable or unexploitable poker. Reproduction and resume instructions
are in the [training guide](../docs/week2-training.md).

This notebook's preview defaults to one epoch; the published matrix fixed three.
